# 合法手がない場合の方策をカスタマイズする

MinimaxPlayer.fallback() をオーバーライドして、合法手がない場合に攻撃技を優先する AI を実装する。

In [1]:
%pip install -q jpoke

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.1.2
[notice] To update, run: C:\Users\tmtmh\Documents\pokemon\jpoke\.venv\Scripts\python.exe -m pip install --upgrade pip


In [ ]:
from jpoke import Battle
from jpoke.players import MinimaxPlayer, RandomPlayer
from jpoke.enums import Command

In [ ]:
class AttackPreferringPlayer(MinimaxPlayer):
    """fallback時（探索できない局面）は攻撃技を優先するAI（fallback()の拡張例）。"""

    def fallback(self, battle: Battle) -> Command:
        commands = battle.available_commands(self)

        def is_attack(command: Command) -> bool:
            # is_regular_move の定義は docs/api/README.md の Command 節を参照
            # （変化技〔まもる等〕も含む点に注意。攻撃技かどうかはmove.is_attackで判定する）
            return command.is_regular_move and battle.command_to_move(self, command).is_attack

        attack_commands = [c for c in commands if is_attack(c)]
        if attack_commands:
            return attack_commands[0]
        # 攻撃技がない場合は既定のランダム選択に委譲する
        return super().fallback(battle)

In [4]:
ai_player = AttackPreferringPlayer("AttackPreferring", max_plies=1, max_nodes=50)
ai_player.add_pokemon("カビゴン", moves=["たくわえる", "のしかかり"])

opponent = RandomPlayer("Opponent")
opponent.add_pokemon("カビゴン", moves=["のしかかり"])

battle = Battle(ai_player, opponent, seed=1)
battle.start()

# 1ターン目: 相手の技が未公開でestimate_opponentも未実装のためfallback()に
# 委譲される。既定のランダム選択なら「まもる」も等確率で選ばれ得るが、この
# 実装では攻撃技「のしかかり」が優先して選ばれる
battle.step()
active = battle.get_active(ai_player)
battle.print_logs()

Turn 1 : Opponent : カビゴン : テラスタル化した（タイプ: ノーマル）
Turn 1 : AttackPreferring : カビゴン : のしかかり PP -1
Turn 1 : Opponent : カビゴン : HP -79 (156/235)
Turn 1 : Opponent : カビゴン : まひが付与された
Turn 1 : Opponent : カビゴン : のしかかり PP -1
Turn 1 : AttackPreferring : カビゴン : HP -110 (125/235)
